# 运行模型

In [ ]:
from argparse import Namespace
from ml.models.mlp import MLP
from ml.models.rnn import RNN
from ml.models.lstm import LSTM
from ml.models.gru import GRU
from ml.models.cnn import CNN
from ml.models.rnn_autoencoder import DualAttentionAutoEncoder

## 超参数

In [ ]:
args = Namespace(
    data_path='../CBL-dataset/LCL-June2015v2_0.csv', # 电力数据集路径
    data_path_test=None, # 测试数据集（可选）
    test_size=0.2, # 验证集比例
    targets=['kwh'], # 目标列：用电量
    num_lags=10, # 用于输入的过去观测值数量

    
    filter_bs=None, # 是否使用单个客户进行训练，将在后续动态更改
    identifier='customer_id', # 标识客户的列名

    nan_constant=0, # 用于转换NaN值的常数
    x_scaler='minmax', # X特征缩放器
    y_scaler='minmax', # y目标缩放器
    outlier_detection=None, # 是否执行异常值处理（flooring and capping）

    
    criterion='mse', # 优化准则，mse或l1
    epochs=10, # 最大训练轮数
    lr=0.001, # 学习率
    optimizer='adam', # 优化器，可以是sgd或adam
    batch_size=128, # 批次大小
    early_stopping=True, # 是否使用早停
    patience=5, # 早停的耐心值（如果指定）
    max_grad_norm=0.0, # 是否裁剪梯度范数
    reg1=0.0, # l1正则化
    reg2=0.0, # l2正则化
    
    plot_history=True, # 绘制损失历史

    cuda=True, # 是否使用GPU
    
    seed=0, # 随机种子（可重现性）

    assign_stats=None, # 是否使用统计量作为外生数据, ["mean", "median", "std", "variance", "kurtosis", "skew"]
    use_time_features=False # 是否使用日期时间特征
)

## 确定输入层神经元的个数

X_train的shape (799714, 10, 1, 1)

X_val的shap    (199686, 10, 1, 1)

y_train的shape (799714, 1)

y_val的shape   (199686, 1)


In [ ]:
def get_input_dims(X_train, exogenous_data_train):
    # 如果模型是 "mlp" (多层感知机)：只能接受“扁平”的一维向量。它会把三维数据 (样本, 时间步, 特征) 的后两个维度乘起来。
    if args.model_name == "mlp":
        input_dim = X_train.shape[1] * X_train.shape[2]

    # 其他 (如 "lstm", "gru")，一次只看“一个时间点”的特征，然后循环看 10 次。
    else:
        input_dim = X_train.shape[2]
    
    # 计算额外辅助特征有多少列
    if exogenous_data_train is not None:
        if len(exogenous_data_train) == 1:
            cid = next(iter(exogenous_data_train.keys()))
            exogenous_dim = exogenous_data_train[cid].shape[1]
        else:
            exogenous_dim = exogenous_data_train["all"].shape[1]
    else:
        exogenous_dim = 0
    
    return input_dim, exogenous_dim # 返回值：输入维度: 1, 外生特征维度: 0

## 定义模型

In [ ]:
def get_model(model: str,
              input_dim: int,
              out_dim: int,
              lags: int = 10,
              exogenous_dim: int = 0,
              seed=0):

    # 多层感知机：最基础的全连接网络          
    if model == "mlp":
        model = MLP(input_dim=input_dim, layer_units=[256, 128, 64], num_outputs=out_dim)

    # 循环神经网络（RNN）：适合处理序列数据，能够捕捉时间序列中的依赖关系
    elif model == "rnn":
        model = RNN(input_dim=input_dim, rnn_hidden_size=128, num_rnn_layers=1, rnn_dropout=0.0,
                    layer_units=[128], num_outputs=out_dim, matrix_rep=True, exogenous_dim=exogenous_dim)

    # 长短期记忆网络（LSTM）：适合处理长序列数据，能够捕捉时间序列中的长期依赖关系
    elif model == "lstm":
        model = LSTM(input_dim=input_dim, lstm_hidden_size=128, num_lstm_layers=1, lstm_dropout=0.0,
                     layer_units=[128], num_outputs=out_dim, matrix_rep=True, exogenous_dim=exogenous_dim)

    # 门控循环单元（GRU）：适合处理序列数据，能够捕捉时间序列中的依赖关系
    elif model == "gru":
        model = GRU(input_dim=input_dim, gru_hidden_size=128, num_gru_layers=1, gru_dropout=0.0,
                    layer_units=[128], num_outputs=out_dim, matrix_rep=True, exogenous_dim=exogenous_dim)

    # 卷积神经网络（CNN）：适合处理图像数据，能够捕捉时间序列中的空间关系
    elif model == "cnn":
        model = CNN(num_features=input_dim, lags=lags, exogenous_dim=exogenous_dim, out_dim=out_dim)

    # 双注意力自动编码器（DA-Encoder-Decoder）：适合处理序列数据，能够捕捉时间序列中的依赖关系
    elif model == "da_encoder_decoder":
        model = DualAttentionAutoEncoder(input_dim=input_dim, architecture="lstm", matrix_rep=True)
    else:
        raise NotImplementedError("Specified model is not implemented. Plese define your own model or choose one from ['mlp', 'rnn', 'lstm', 'gru', 'cnn', 'da_encoder_decoder']")
    return model

## 使用模型

In [ ]:
# 定义模型
args.model_name = "lstm"

# 获取输入维度
input_dim, exogenous_dim = get_input_dims(X_train, exogenous_data_train)

print(f"输入维度: {input_dim}, 外生特征维度: {exogenous_dim}") # 输入维度: 1, 外生特征维度: 0
print(f"输出维度: {y_train.shape[1]}")                       # 输出维度: 1（只有kwh一个目标）

# 创建模型
model = get_model(model=args.model_name,
                  input_dim=input_dim,
                  out_dim=y_train.shape[1],  # 电力数据只有1个目标（kwh）
                  lags=args.num_lags,
                  exogenous_dim=exogenous_dim,
                  seed=args.seed)

## 训练过程（集中式）

In [ ]:
def fit(model, 
        X_train, y_train, X_val, y_val, 
        exogenous_data_train=None, 
        exogenous_data_val=None, 
        idxs=[0], # 目标在X中的索引，对于电力数据只有kwh一个目标，所以是[0]
        log_per=1):
    
    # 准备外生变量
    if exogenous_data_train is not None and len(exogenous_data_train) > 1:
        exogenous_data_train = exogenous_data_train["all"]
        exogenous_data_val = exogenous_data_val["all"]
    elif exogenous_data_train is not None and len(exogenous_data_train) == 1:
        cid = next(iter(exogenous_data_train.keys()))
        exogenous_data_train = exogenous_data_train[cid]
        exogenous_data_val = exogenous_data_val[cid]
    else:
        exogenous_data_train = None
        exogenous_data_val = None
    num_features = len(X_train[0][0])
    
    # 构建Pytorch数据加载器
    train_loader = to_torch_dataset(X_train, y_train,
                                    num_lags=args.num_lags,
                                    num_features=num_features,
                                    exogenous_data=exogenous_data_train,
                                    indices=idxs,
                                    batch_size=args.batch_size, 
                                    shuffle=False)

    val_loader = to_torch_dataset(X_val, y_val, 
                                  num_lags=args.num_lags,
                                  num_features=num_features,
                                  exogenous_data=exogenous_data_val,
                                  indices=idxs,
                                  batch_size=args.batch_size,
                                  shuffle=False)
    
    # 点火启动
    model = train(model, 
                  train_loader, val_loader,
                  epochs=args.epochs,
                  optimizer=args.optimizer, lr=args.lr,
                  criterion=args.criterion,
                  early_stopping=args.early_stopping,
                  patience=args.patience,
                  plot_history=args.plot_history, 
                  device=device, log_per=log_per)
    
    return model


# 调用
trained_model = fit(model, X_train, y_train, X_val, y_val)

## 训练过程（联邦式）

```
X_train的shape (799714, 10, 1, 1)
X_val的shap    (199686, 10, 1, 1)
y_train的shape (799714, 1)
y_val的shape   (199686, 1)


用户MAC000007的shape (20027, 10, 1, 1)
用户MAC000007的shape (4999, 10, 1, 1)
用户MAC000007的shape (20027, 1)
用户MAC000007的shape (4999, 1)
```

In [ ]:
def fit(model, X_train, y_train, X_val, y_val, 
        exogenous_data_train=None, exogenous_data_val=None, 
        idxs=[0],   # 目标在X中的索引，对于电力数据只有kwh一个目标，所以是[0]
        log_per=1,
        client_creation_fn = None, # client specification
        local_train_params=None, # local params
        aggregation_params=None, # aggregation params
        use_carbontracker=True
       ):

    # create_regression_client 是一个工厂函数，负责把模型、数据和训练算法封装成一个独立的客户端对象。
    if client_creation_fn is None:
        client_creation_fn = create_regression_client


    # # 配置每个客户端在本地训练时用的参数（轮数、优化器、学习率、损失函数等）
    if local_train_params is None:
        local_train_params = {
            "epochs": args.epochs, "optimizer": args.optimizer, "lr": args.lr,
            "criterion": args.criterion, "early_stopping": args.local_early_stopping,
            "patience": args.local_patience, "device": device
        }
    
    # 为每个客户端准备独立的数据加载器
    train_loaders, val_loaders = [], []
    
    # 为每个客户端分配数据
    for client in X_train:
        if client == "all":
            continue
        # # 如果有外部变量，提取该客户端对应的部分
        if exogenous_data_train is not None:
            tmp_exogenous_data_train = exogenous_data_train[client]
            tmp_exogenous_data_val = exogenous_data_val[client]
        else:
            tmp_exogenous_data_train = None
            tmp_exogenous_data_val = None
    
        # 计算每个客户端的特征数量
        num_features = len(X_train[client][0][0])
        
        # 将该客户端的NumPy数组打包成PyTorch的DataLoader（用于分批次训练）
        train_loaders.append(
            to_torch_dataset(
                X_train[client], y_train[client],
                num_lags=args.num_lags,
                num_features=num_features,
                exogenous_data=tmp_exogenous_data_train,
                indices=idxs,
                batch_size=args.batch_size,
                shuffle=False
            )
        )
        val_loaders.append(
            to_torch_dataset(
                X_val[client], y_val[client],
                num_lags=args.num_lags,
                num_features=num_features,
                exogenous_data=tmp_exogenous_data_val,
                indices=idxs,
                batch_size=args.batch_size,
                shuffle=False
            )
            
        )
        
    # 实例化客户端对象
    cids = [k for k in X_train.keys() if k != "all"]
    # 把刚才准备好的模型、本地数据、本地参数全部封装进“客户端”对象里
    clients = [
        client_creation_fn(
            cid=cid, # client id
            model=model, # the global model
            train_loader=train_loader, # the local train loader
            test_loader=val_loader, # the local val loader
            local_params=local_train_params # local parameters
        )
        for cid, train_loader, val_loader in zip(cids, train_loaders, val_loaders)
    ]
    
    # 建立服务端与客户端的通信桥梁
    client_proxies = [
        SimpleClientProxy(cid, client) for cid, client in zip(cids, clients)
    ]
    
    # 实例化中央服务端对象
    server = Server(
        client_proxies=client_proxies, # the client representations
        aggregation=args.aggregation, # the aggregation algorithm
        aggregation_params=aggregation_params, # aggregation specific params
        local_params_fn=None, # we can change the local params on demand
    )
    # Note that the client manager instance will be initialized automatically. You can define your own client manager.

    # 正式开始联邦训练！返回最终聚合成的全局参数 model_params 和 历史指标 history
    model_params, history = server.fit(args.fl_rounds, args.fraction, use_carbontracker=use_carbontracker)
    
    # 将服务器传回来的“纯数字列表格式”的参数，重新转换回PyTorch模型要求的字典格式
    params_dict = zip(model.state_dict().keys(), model_params)
    state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})

    # 深拷贝一个模型，并把刚才生成的全局参数加载进去，形成最终的“成品模型”
    model = copy.deepcopy(model)
    model.load_state_dict(state_dict, strict=True)
    
    # 返回最终的全局模型和训练历史数据
    return model, history

## 设置联邦学习中每个客户端的参数

In [ ]:
# federated local params
local_train_params = {"epochs": args.epochs, "optimizer": args.optimizer, "lr": args.lr,
                      "criterion": args.criterion, "early_stopping": args.local_early_stopping,
                      "patience": args.local_patience, "device": device
                      }

## 开训

In [ ]:
global_model, history = fit(
    model,
    client_X_train,
    client_y_train, 
    client_X_val, 
    client_y_val, 
    local_train_params=local_train_params
)